In [3]:
# 1️⃣ Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor


In [4]:
file_path = "../Datasets/master/propwise_master.csv"
df = pd.read_csv(file_path)
print(df.head())

# Quick look at data
df.head()

   SALE_YEAR SALE_QUARTER  SALE_MONTH  POSTAL_CODE              ADDRESS  \
0       2020           Q3           8     543204.0  204C COMPASSVALE DR   
1       2020           Q3           8     543204.0  204C COMPASSVALE DR   
2       2020           Q3           8     543204.0  204C COMPASSVALE DR   
3       2020           Q3           8     543204.0  204C COMPASSVALE DR   
4       2020           Q3           8     543204.0  204C COMPASSVALE DR   

   RESALE_INDEX  RESALE_PRICE  MIN_SELLING_PRICE  MEDIAN_RESALE_PRICE  \
0         134.0        420000                NaN             560000.0   
1         134.0        420000                NaN                  NaN   
2         134.0        420000                NaN             410000.0   
3         134.0        420000                NaN             475000.0   
4         134.0        420000                NaN             590000.0   

   MAX_SELLING_PRICE  ... NEAREST_PARK_DISTANCE_M PARKS_WITHIN_1KM  \
0                NaN  ...               

,SALE_YEAR,SALE_QUARTER,SALE_MONTH,POSTAL_CODE,ADDRESS,RESALE_INDEX,RESALE_PRICE,MIN_SELLING_PRICE,MEDIAN_RESALE_PRICE,MAX_SELLING_PRICE,...,NEAREST_PARK_DISTANCE_M,PARKS_WITHIN_1KM,NEAREST_PRESCHOOL_M,PRESCHOOLS_WITHIN_1KM,NEAREST_CLINIC_M,CLINICS_WITHIN_500M,NEAREST_PHARMACY_M,NEAREST_HOSPITAL_M,HEALTHCARE_ACCESSIBILITY_SCORE,NEAREST_MALL_M
0,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,560000.0,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163
1,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,NaN,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163
2,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,410000.0,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163
3,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,475000.0,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163
4,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,590000.0,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163


In [5]:
# 3️⃣ Drop leakage / high-risk columns
leakage_cols = ["MIN_SELLING_PRICE", "MEDIAN_RESALE_PRICE", "MAX_SELLING_PRICE", "RESALE_INDEX"]
df = df.drop(columns=[col for col in leakage_cols if col in df.columns])
print("\nDropped leakage columns:", leakage_cols)


Dropped leakage columns: ['MIN_SELLING_PRICE', 'MEDIAN_RESALE_PRICE', 'MAX_SELLING_PRICE', 'RESALE_INDEX']


In [6]:
# 4️⃣ Set target column
target_column = "RESALE_PRICE"  # numeric target
df = df.dropna(subset=[target_column])  # remove rows with missing target

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]

In [7]:
# 5️⃣ Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Drop columns that are too unique / IDs
for col in ['ADDRESS', 'NEAREST_MRT_NAME']:
    if col in categorical_cols:
        categorical_cols.remove(col)

# Fill missing values
for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())
for col in categorical_cols:
    X[col] = X[col].fillna('missing')

In [8]:
# 6️⃣ Feature engineering
# Example: property age
if 'YEAR_COMPLETED' in numeric_cols:
    X['PROPERTY_AGE'] = 2026 - X['YEAR_COMPLETED']
    numeric_cols.append('PROPERTY_AGE')
    numeric_cols.remove('YEAR_COMPLETED')

# 7️⃣ Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [9]:
# 8️⃣ Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# 9️⃣ Build pipeline with XGBoost
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

In [11]:
# 1️⃣0️⃣ Train model
print("\nTraining model...")
model.fit(X_train, y_train)
print("Model training complete!")


Training model...
Model training complete!


In [12]:

# predictions
y_pred = model.predict(X_test)

# Evaluation
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)  # manually compute RMSE

print("R2 Score:", r2)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)


R2 Score: 0.9724165797233582
MAE: 20905.490234375
MSE: 803885312.0
RMSE: 28352.87131843969


## Testing

In [13]:
import pandas as pd

# Create a sample input for prediction
sample_data = {
    "SALE_YEAR": [2026],
    "SALE_QUARTER": ["Q1"],
    "SALE_MONTH": [1],
    "POSTAL_CODE": [822000],        # Punggol postal
    "ADDRESS": ["dummy address"],    # will be ignored in preprocessing
    "TOWN": ["PUNGGOL"],
    "FLAT_TYPE": ["2 ROOM"],
    "STOREY_RANGE": ["01 TO 03"],
    "FLAT_MODEL": ["Model A"],       # pick a common flat model
    "YEAR_COMPLETED": [2020],
    "MAX_FLOOR_LEVEL": [5],
    "REMAINING_LEASE_YEARS": [95],
    "LATITUDE": [1.403],             # example coordinates
    "LONGITUDE": [103.902],
    "OVERALL_AMENITY_SCORE": [50],
    "NEAREST_MRT_NAME": ["PUNGGOL"],
    "NEAREST_MRT_DISTANCE_M": [200],
    "MRT_WITHIN_500M": [1],
    "MRT_WITHIN_1KM": [1],
    "NEAREST_IS_NS_LINE": [0],
    "NEAREST_IS_EW_LINE": [0],
    "NEAREST_GYM_DISTANCE_M": [500],
    "GYMS_WITHIN_1KM": [3],
    "NEAREST_HAWKER_DISTANCE_M": [300],
    "HAWKERS_WITHIN_500M": [2],
    "HAWKERS_WITHIN_1KM": [5],
    "TOTAL_FOOD_STALLS_WITHIN_1KM": [10],
    "NEAREST_SUPERMARKET_DISTANCE_M": [400],
    "SUPERMARKETS_WITHIN_500M": [1],
    "SUPERMARKETS_WITHIN_1KM": [3],
    "SHOPPING_CONVENIENCE_SCORE": [50],
    "NEAREST_PARK_DISTANCE_M": [150],
    "PARKS_WITHIN_1KM": [2],
    "NEAREST_PRESCHOOL_M": [100],
    "PRESCHOOLS_WITHIN_1KM": [5],
    "NEAREST_CLINIC_M": [300],
    "CLINICS_WITHIN_500M": [1],
    "NEAREST_PHARMACY_M": [200],
    "NEAREST_HOSPITAL_M": [500],
    "HEALTHCARE_ACCESSIBILITY_SCORE": [60],
    "NEAREST_MALL_M": [800]
}

sample_df = pd.DataFrame(sample_data)

# Create property age (feature engineering)
sample_df['PROPERTY_AGE'] = 2026 - sample_df['YEAR_COMPLETED']
sample_df = sample_df.drop(columns=['YEAR_COMPLETED'])

# Predict resale price
predicted_price = model.predict(sample_df)
print(f"Predicted resale price for 2-room flat in Punggol (2026): ${predicted_price[0]:,.0f}")


Predicted resale price for 2-room flat in Punggol (2026): $363,871
